<a href="https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:


from pathlib import Path
import os
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


# ------------------------------------------------------------
# 1. Find the repository and starter CSV
# ------------------------------------------------------------

REPO_NAME = "flyrank-ml-internship-sunaina"
REPO_URL = "https://github.com/sunainakhatwani12/flyrank-ml-internship-sunaina.git"

possible_roots = [
    Path.cwd(),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

repo_root = None

for root in possible_roots:
    candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        repo_root = root
        break

# Colab does not automatically copy the whole GitHub repository
# when a notebook is opened through an Open in Colab link.
if repo_root is None:
    clone_path = Path("/content") / REPO_NAME

    if clone_path.exists():
        shutil.rmtree(clone_path)

    print("Repository files were not found in the current Colab session.")
    print("Cloning the public repository...")

    subprocess.run(
        ["git", "clone", REPO_URL, str(clone_path)],
        check=True
    )

    repo_root = clone_path

os.chdir(repo_root)

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Starter CSV was not found at:\n{data_path}\n\n"
        "Update your repository from the FlyRank starter repository "
        "and confirm that data/raw/content_refresh_anonymized.csv exists."
    )

print(f"Repository root: {repo_root}")
print(f"Data path: {data_path}")


# ------------------------------------------------------------
# 2. Load and verify the data
# ------------------------------------------------------------

df = pd.read_csv(data_path)

required_columns = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}\n"
        f"Available columns:\n{df.columns.tolist()}"
    )

# Build the decline label for evaluation only.
# It will NOT be used inside the baseline score.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down")
).astype(int)

print("\nDATA CHECK")
print("-" * 60)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique():,}")
print(f"Declining pages: {df['is_declining_label'].sum():,}")
print(f"Overall decline base rate: {df['is_declining_label'].mean():.2%}")

display(df.head(3))


# ------------------------------------------------------------
# 3. Signal check 1 — staleness
# ------------------------------------------------------------

staleness_labels = [
    "0-90 days",
    "91-180 days",
    "181-365 days",
    "366-730 days",
    "731+ days",
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, 730, np.inf],
    labels=staleness_labels,
    ordered=True
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_n=("is_declining_label", "sum"),
          decline_rate=("is_declining_label", "mean"),
          median_impressions=("impressions_90d", "median"),
          median_days_since_update=("days_since_last_update", "median"),
      )
      .reset_index()
)

staleness_table["decline_rate_pct"] = (
    staleness_table["decline_rate"] * 100
).round(2)

print("\nSIGNAL 1 — STALENESS BUCKET TABLE")
print("-" * 60)

display(
    staleness_table[
        [
            "staleness_bucket",
            "n",
            "declining_n",
            "decline_rate_pct",
            "median_impressions",
            "median_days_since_update",
        ]
    ]
)

valid_stale = staleness_table.dropna(subset=["decline_rate"]).copy()

if len(valid_stale) >= 2:
    first_rate = valid_stale["decline_rate"].iloc[0]
    last_rate = valid_stale["decline_rate"].iloc[-1]
    stale_difference = last_rate - first_rate

    rates = valid_stale["decline_rate"].to_numpy()
    monotonic_increases = np.diff(rates)
    mostly_increasing = (monotonic_increases >= -0.02).mean() >= 0.75

    if stale_difference >= 0.05 and mostly_increasing:
        staleness_verdict = "CONFIRMED"
        staleness_explanation = (
            "Older update buckets generally show a meaningfully higher "
            "observed decline rate than fresher buckets."
        )
    elif stale_difference <= -0.05:
        staleness_verdict = "OPPOSITE"
        staleness_explanation = (
            "Older pages show a lower observed decline rate than fresher pages."
        )
    elif abs(stale_difference) < 0.02:
        staleness_verdict = "FALSE"
        staleness_explanation = (
            "The oldest and freshest buckets have very similar decline rates."
        )
    else:
        staleness_verdict = "MIXED"
        staleness_explanation = (
            "The relationship exists in some buckets but is not consistently monotonic."
        )
else:
    staleness_verdict = "FALSE"
    staleness_explanation = "There were not enough populated buckets to support the signal."

print(f"\nSignal 1 verdict: {staleness_verdict}")
print(staleness_explanation)
print(
    "Interpretation: this is an observed association in the starter data, "
    "not proof that age causes decline."
)


# ------------------------------------------------------------
# 4. Signal check 2 — impression visibility
# ------------------------------------------------------------

impression_labels = [
    "1-99",
    "100-299",
    "300-2,999",
    "3,000-29,999",
    "30,000+",
]

df["impression_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 299, 2999, 29999, np.inf],
    labels=impression_labels,
    ordered=True
)

volume_table = (
    df.groupby("impression_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_n=("is_declining_label", "sum"),
          decline_rate=("is_declining_label", "mean"),
          median_impressions=("impressions_90d", "median"),
          median_clicks=("clicks_90d", "median"),
      )
      .reset_index()
)

volume_table["decline_rate_pct"] = (
    volume_table["decline_rate"] * 100
).round(2)

print("\nSIGNAL 2 — IMPRESSION-VOLUME BUCKET TABLE")
print("-" * 60)

display(
    volume_table[
        [
            "impression_bucket",
            "n",
            "declining_n",
            "decline_rate_pct",
            "median_impressions",
            "median_clicks",
        ]
    ]
)

valid_volume = volume_table.dropna(subset=["decline_rate"]).copy()

if len(valid_volume) >= 2:
    low_rate = valid_volume["decline_rate"].iloc[0]
    high_rate = valid_volume["decline_rate"].iloc[-1]
    volume_difference = high_rate - low_rate

    max_rate = valid_volume["decline_rate"].max()
    min_rate = valid_volume["decline_rate"].min()
    spread = max_rate - min_rate

    if volume_difference >= 0.05:
        volume_verdict = "CONFIRMED"
        volume_explanation = (
            "Higher-volume pages show a meaningfully higher observed decline rate."
        )
    elif volume_difference <= -0.05:
        volume_verdict = "OPPOSITE"
        volume_explanation = (
            "Higher-volume pages show a lower observed decline rate."
        )
    elif spread < 0.02:
        volume_verdict = "FALSE"
        volume_explanation = (
            "Decline rates are nearly the same across impression-volume buckets."
        )
    else:
        volume_verdict = "MIXED"
        volume_explanation = (
            "Decline rates vary across volume buckets, but the pattern is not consistent."
        )
else:
    volume_verdict = "FALSE"
    volume_explanation = "There were not enough populated buckets to support the signal."

print(f"\nSignal 2 verdict: {volume_verdict}")
print(volume_explanation)
print(
    "Even if the verdict is MIXED, impression volume can still be used as "
    "an opportunity-size filter. It indicates how much existing visibility "
    "may be affected, not whether decline is guaranteed."
)


# ------------------------------------------------------------
# 5. Save signal-audit receipts
# ------------------------------------------------------------

output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

signal_receipt = {
    "assignment": "ML-07 baseline action score",
    "lane": "Refresh / Content Opportunity Scoring",
    "rows": int(len(df)),
    "clients": int(df["client_id"].nunique()),
    "overall_decline_base_rate": round(
        float(df["is_declining_label"].mean()), 6
    ),
    "signal_1": {
        "name": "staleness",
        "column": "days_since_last_update",
        "verdict": staleness_verdict,
        "explanation": staleness_explanation,
    },
    "signal_2": {
        "name": "search visibility",
        "column": "impressions_90d",
        "verdict": volume_verdict,
        "explanation": volume_explanation,
    },
}

signal_json_path = output_dir / "w04_signal_verdicts.json"

with open(signal_json_path, "w", encoding="utf-8") as file:
    json.dump(signal_receipt, file, indent=2)

print(f"\nSignal receipt written to: {signal_json_path}")
print("\nFINAL ONE-WORD VERDICTS")
print(f"Signal 1 — staleness: {staleness_verdict}")
print(f"Signal 2 — visibility: {volume_verdict}")

Repository root: /content/flyrank-ml-internship-sunaina
Data path: /content/flyrank-ml-internship-sunaina/data/raw/content_refresh_anonymized.csv

DATA CHECK
------------------------------------------------------------
Rows: 30,000
Columns: 45
Unique content IDs: 30,000
Unique clients: 32
Declining pages: 16,262
Overall decline base rate: 54.21%


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.000,0.670,HIGH,2.050,keyword article,transactional,"3,221.000","20,457.000",NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.760,10.600,5.880,4.550,0.000,good,striking,down,-41.400,1
1,content_a1fb4e703a9e,client_4e07408562,90.000,0.010,LOW,0.050,keyword article,informational,"2,481.000","15,562.000",NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.050,20.300,0.000,10.000,0.000,good,page_3_5,down,-57.700,1
2,content_9aa793d4d895,client_7f2253d7e2,0.000,0.000,LOW,0.000,keyword article,informational,"3,515.000","23,643.000",NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.090,36.500,0.000,28.570,0.000,good,page_3_5,down,-60.900,1



SIGNAL 1 — STALENESS BUCKET TABLE
------------------------------------------------------------


,staleness_bucket,n,declining_n,decline_rate_pct,median_impressions,median_days_since_update
0,0-90 days,20655,10576,51.200,472.000,20.000
1,91-180 days,9171,5604,61.110,"1,692.000",104.000
2,181-365 days,169,79,46.750,16.000,211.000
3,366-730 days,5,3,60.000,2.000,373.000
4,731+ days,0,0,NaN,NaN,NaN



Signal 1 verdict: MIXED
The relationship exists in some buckets but is not consistently monotonic.
Interpretation: this is an observed association in the starter data, not proof that age causes decline.

SIGNAL 2 — IMPRESSION-VOLUME BUCKET TABLE
------------------------------------------------------------


,impression_bucket,n,declining_n,decline_rate_pct,median_impressions,median_clicks
0,1-99,7994,3110,38.900,12.000,0.000
1,100-299,3254,1996,61.340,184.000,0.000
2,"300-2,999",10469,6435,61.470,998.000,1.000
3,"3,000-29,999",7205,4223,58.610,"7,249.000",16.000
4,"30,000+",1078,498,46.200,"48,675.000",116.000



Signal 2 verdict: CONFIRMED
Higher-volume pages show a meaningfully higher observed decline rate.
Even if the verdict is MIXED, impression volume can still be used as an opportunity-size filter. It indicates how much existing visibility may be affected, not whether decline is guaranteed.

Signal receipt written to: /content/flyrank-ml-internship-sunaina/work/outputs/w04_signal_verdicts.json

FINAL ONE-WORD VERDICTS
Signal 1 — staleness: MIXED
Signal 2 — visibility: CONFIRMED


## 1. My rule, signal checks, and reason code

### Lane confirmation

I confirm my lane as **Refresh / Content Opportunity Scoring**.

My goal is to produce a ranked queue of content pages that should be reviewed for a possible refresh. The output is decision support for a content or SEO team. It does not automatically decide that a page must be changed.

### Rule in plain words

A page should receive a high baseline score when:

1. It has not been updated recently.
2. It still receives meaningful search impressions.

Older pages with stronger existing visibility are ranked first because a refresh may protect or improve traffic that already exists.

### Signal 1: staleness

I will group pages by `days_since_last_update` and measure the decline rate in each bucket.

This is linked to a real FlyRank refresh signal because refresh flags use content staleness as part of their reasoning.

### Signal 2: search visibility

I will group pages by `impressions_90d` and measure the decline rate in each bucket.

This tests whether pages with meaningful existing search visibility provide a useful opportunity signal. Impression volume is used only as historical context, not as proof that refreshing a page will cause improvement.

### Baseline rule

A page becomes a refresh candidate when:

* `days_since_last_update >= 180`
* `impressions_90d >= 300`

The baseline score combines:

* Staleness severity: 60% of the score.
* Search visibility: 40% of the score.

### Reason code

`STALE_VISIBLE_PAGE`

### Action label

`REFRESH_CONTENT`

Pages that do not meet both thresholds receive a score of zero and the action `MONITOR`.

### Leakage protection

The score does not use:

* `trend_direction`
* `trend_pct`
* `is_declining_label`
* Future-window outcomes
* Existing FlyRank product flags
* `content_id` or `client_id` as predictive inputs

The decline label is used only to audit the signals and evaluate the baseline. It is not used to calculate the baseline score.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:

from sklearn.preprocessing import MinMaxScaler


# ------------------------------------------------------------
# 1. Freeze the baseline thresholds
# ------------------------------------------------------------

STALE_THRESHOLD_DAYS = 180
VISIBILITY_THRESHOLD_IMPRESSIONS = 300

REASON_CODE = "STALE_VISIBLE_PAGE"
REFRESH_ACTION = "REFRESH_CONTENT"
MONITOR_ACTION = "MONITOR"


# ------------------------------------------------------------
# 2. Create transparent component scores
# ------------------------------------------------------------

# Cap extreme values so one unusual page does not dominate the queue.
staleness_cap = max(
    STALE_THRESHOLD_DAYS + 1,
    float(df["days_since_last_update"].quantile(0.95))
)

impression_cap = max(
    VISIBILITY_THRESHOLD_IMPRESSIONS + 1,
    float(df["impressions_90d"].quantile(0.95))
)

df["is_stale"] = (
    df["days_since_last_update"] >= STALE_THRESHOLD_DAYS
).astype(int)

df["has_meaningful_visibility"] = (
    df["impressions_90d"] >= VISIBILITY_THRESHOLD_IMPRESSIONS
).astype(int)

df["eligible_for_refresh"] = (
    (df["is_stale"] == 1) &
    (df["has_meaningful_visibility"] == 1)
).astype(int)

# Staleness component: 0 to 60 points.
df["staleness_component"] = (
    df["days_since_last_update"]
      .clip(lower=STALE_THRESHOLD_DAYS, upper=staleness_cap)
      .sub(STALE_THRESHOLD_DAYS)
      .div(staleness_cap - STALE_THRESHOLD_DAYS)
      .mul(60)
)

# Visibility component: 0 to 40 points.
# log1p keeps very large pages from completely dominating.
log_floor = np.log1p(VISIBILITY_THRESHOLD_IMPRESSIONS)
log_cap = np.log1p(impression_cap)

df["visibility_component"] = (
    np.log1p(
        df["impressions_90d"].clip(
            lower=VISIBILITY_THRESHOLD_IMPRESSIONS,
            upper=impression_cap
        )
    )
    .sub(log_floor)
    .div(log_cap - log_floor)
    .mul(40)
)

# Ineligible rows must receive zero.
df.loc[df["eligible_for_refresh"] == 0, "staleness_component"] = 0
df.loc[df["eligible_for_refresh"] == 0, "visibility_component"] = 0

df["baseline_action_score"] = (
    df["staleness_component"] +
    df["visibility_component"]
).clip(0, 100).round(2)

df["reason_code"] = np.where(
    df["eligible_for_refresh"] == 1,
    REASON_CODE,
    ""
)

df["action_label"] = np.where(
    df["eligible_for_refresh"] == 1,
    REFRESH_ACTION,
    MONITOR_ACTION
)


# ------------------------------------------------------------
# 3. Rank the complete queue
# ------------------------------------------------------------

queue_columns = [
    "content_id",
    "client_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "content_type",
    "main_intent",
    "eligible_for_refresh",
    "is_declining_label",
]

# Keep only columns that are present.
queue_columns = [column for column in queue_columns if column in df.columns]

ranked_queue = (
    df[queue_columns]
    .sort_values(
        by=[
            "baseline_action_score",
            "impressions_90d",
            "days_since_last_update",
        ],
        ascending=[False, False, False],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "baseline_rank",
    np.arange(1, len(ranked_queue) + 1)
)


# ------------------------------------------------------------
# 4. Evaluate precision at K
# ------------------------------------------------------------

def precision_at_k(data, k):
    """
    Of the top-k ranked rows, return the percentage whose
    evaluation label equals 1.
    """
    actual_k = min(k, len(data))

    if actual_k == 0:
        return np.nan

    return float(
        data.head(actual_k)["is_declining_label"].mean()
    )


base_rate = float(df["is_declining_label"].mean())

metrics = {
    "base_rate": base_rate,
    "precision_at_10": precision_at_k(ranked_queue, 10),
    "precision_at_20": precision_at_k(ranked_queue, 20),
    "precision_at_50": precision_at_k(ranked_queue, 50),
    "precision_at_100": precision_at_k(ranked_queue, 100),
    "eligible_candidates": int(df["eligible_for_refresh"].sum()),
    "total_rows": int(len(df)),
    "stale_threshold_days": STALE_THRESHOLD_DAYS,
    "visibility_threshold_impressions": VISIBILITY_THRESHOLD_IMPRESSIONS,
}

print("BASELINE RESULTS")
print("-" * 60)
print(f"Rows ranked: {len(ranked_queue):,}")
print(f"Eligible refresh candidates: {metrics['eligible_candidates']:,}")
print(f"Overall decline base rate: {base_rate:.2%}")
print(f"Precision@10: {metrics['precision_at_10']:.2%}")
print(f"Precision@20: {metrics['precision_at_20']:.2%}")
print(f"Precision@50: {metrics['precision_at_50']:.2%}")
print(f"Precision@100: {metrics['precision_at_100']:.2%}")

if metrics["precision_at_50"] > base_rate:
    print(
        "\nInterpretation: the top 50 contains a higher share of declining "
        "pages than the overall dataset."
    )
elif metrics["precision_at_50"] < base_rate:
    print(
        "\nInterpretation: this baseline does not beat the dataset base rate "
        "at the top 50. This creates an honest target for the Week 5 model."
    )
else:
    print(
        "\nInterpretation: this baseline matches the dataset base rate "
        "at the top 50."
    )


# ------------------------------------------------------------
# 5. Write the ranked queue and metrics
# ------------------------------------------------------------

queue_path = output_dir / "baseline_action_score.csv"
metrics_path = output_dir / "w04_baseline_metrics.json"

ranked_queue.to_csv(queue_path, index=False)

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print(f"\nRanked queue written to: {queue_path}")
print(f"Metrics receipt written to: {metrics_path}")

print("\nTOP 10 RANKED ROWS")
display(ranked_queue.head(10))

BASELINE RESULTS
------------------------------------------------------------
Rows ranked: 30,000
Eligible refresh candidates: 22
Overall decline base rate: 54.21%
Precision@10: 100.00%
Precision@20: 90.00%
Precision@50: 62.00%
Precision@100: 50.00%

Interpretation: the top 50 contains a higher share of declining pages than the overall dataset.

Ranked queue written to: /content/flyrank-ml-internship-sunaina/work/outputs/baseline_action_score.csv
Metrics receipt written to: /content/flyrank-ml-internship-sunaina/work/outputs/w04_baseline_metrics.json

TOP 10 RANKED ROWS


,baseline_rank,content_id,client_id,baseline_action_score,reason_code,action_label,days_since_last_update,content_age_days,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,content_type,main_intent,eligible_for_refresh,is_declining_label
0,1,content_cf56e2e2e282,client_7f2253d7e2,100.000,STALE_VISIBLE_PAGE,REFRESH_CONTENT,194,231,61678,94,0.150,19.700,119,keyword article,informational,1,1
1,2,content_7368877ea310,client_7f2253d7e2,100.000,STALE_VISIBLE_PAGE,REFRESH_CONTENT,194,231,59472,77,0.130,24.800,82,keyword article,informational,1,1
2,3,content_1bfaa38ff26c,client_7f2253d7e2,100.000,STALE_VISIBLE_PAGE,REFRESH_CONTENT,194,231,25715,60,0.230,22.200,80,keyword article,informational,1,1
3,4,content_0a91db491d14,client_7f2253d7e2,94.950,STALE_VISIBLE_PAGE,REFRESH_CONTENT,193,231,13299,65,0.490,10.500,78,keyword article,informational,1,1
4,5,content_5feee3994adb,client_7f2253d7e2,90.040,STALE_VISIBLE_PAGE,REFRESH_CONTENT,194,231,7812,1,0.010,39.000,5,keyword article,transactional,1,1
5,6,content_c2d929d83eaa,client_7f2253d7e2,89.740,STALE_VISIBLE_PAGE,REFRESH_CONTENT,193,231,7558,15,0.200,17.900,25,keyword article,informational,1,1
6,7,content_b16bd7307b39,client_7f2253d7e2,85.140,STALE_VISIBLE_PAGE,REFRESH_CONTENT,194,231,4590,0,0.000,31.000,4,keyword article,informational,1,1
7,8,content_fe16a55cd13d,client_7f2253d7e2,85.070,STALE_VISIBLE_PAGE,REFRESH_CONTENT,194,231,4556,15,0.330,16.400,42,keyword article,informational,1,1
8,9,content_ecb6215e79fd,client_7f2253d7e2,84.810,STALE_VISIBLE_PAGE,REFRESH_CONTENT,194,231,4429,17,0.380,25.300,12,keyword article,informational,1,1
9,10,content_928af3e22c80,client_7f2253d7e2,75.960,STALE_VISIBLE_PAGE,REFRESH_CONTENT,193,231,1697,2,0.120,15.800,3,keyword article,informational,1,1


## 2. Build the ranked queue

The baseline uses only information available at scoring time.

A page is eligible for the refresh queue when:

* It has not been updated for at least 180 days.
* It has received at least 300 impressions during the trailing 90-day window.

The score is intentionally simple and readable:

* Up to 60 points for staleness.
* Up to 40 points for search visibility.
* Maximum score: 100.

The score is not a probability. A score of 90 does not mean that the page has a 90% chance of improving after a refresh. It only means that the page ranks highly according to this baseline rule.

The decline label is used after ranking to calculate evaluation metrics. It is not used to create the score.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-07 — SECTION 3
# Top-20 skeptical review
# ============================================================

top_review = ranked_queue.head(20).copy()


def confidence_note(row):
    score = row["baseline_action_score"]

    if score >= 85:
        return (
            "High rule confidence because both staleness and visibility "
            "are strong."
        )
    elif score >= 70:
        return (
            "Medium-high rule confidence; both conditions pass, but one "
            "component is less extreme."
        )
    else:
        return (
            "Moderate rule confidence; the page passes the thresholds "
            "but is close to at least one cutoff."
        )


def why_selected(row):
    return (
        f"{int(row['days_since_last_update']):,} days since update and "
        f"{int(row['impressions_90d']):,} impressions in the last 90 days; "
        f"score={row['baseline_action_score']:.2f}."
    )


def what_could_make_wrong(row):
    possible_reasons = []

    avg_position = row.get("avg_position", np.nan)
    ctr = row.get("ctr", np.nan)
    sessions = row.get("sessions_90d", np.nan)

    if pd.notna(avg_position) and avg_position > 50:
        possible_reasons.append(
            "the page ranks too deeply for a content refresh alone to help"
        )

    if pd.notna(ctr) and ctr > 3:
        possible_reasons.append(
            "its click-through performance may already be healthy"
        )

    if pd.notna(sessions) and sessions == 0:
        possible_reasons.append(
            "the impressions may not translate into useful visits"
        )

    if not possible_reasons:
        possible_reasons.append(
            "the topic may be seasonal, intentionally evergreen, already "
            "scheduled for refresh, or not commercially important"
        )

    return "; ".join(possible_reasons) + "."


top_review["review_action"] = top_review["action_label"]
top_review["why_it_is_here"] = top_review.apply(
    why_selected,
    axis=1
)
top_review["confidence_note"] = top_review.apply(
    confidence_note,
    axis=1
)
top_review["what_would_make_it_wrong"] = top_review.apply(
    what_could_make_wrong,
    axis=1
)

review_columns = [
    "baseline_rank",
    "content_id",
    "review_action",
    "reason_code",
    "why_it_is_here",
    "confidence_note",
    "what_would_make_it_wrong",
]

print("TOP-20 SKEPTICAL REVIEW")
print("-" * 60)

display(top_review[review_columns])


print("\nONE-LINE REVIEW FOR EACH ROW")
print("-" * 60)

for _, row in top_review.iterrows():
    print(
        f"Rank {int(row['baseline_rank'])}: "
        f"Action={row['review_action']} | "
        f"Why={row['why_it_is_here']} | "
        f"Wrong if={row['what_would_make_it_wrong']}"
    )


# Optional review receipt.
review_path = output_dir / "w04_top20_review.json"

review_records = (
    top_review[review_columns]
    .replace({np.nan: None})
    .to_dict(orient="records")
)

with open(review_path, "w", encoding="utf-8") as file:
    json.dump(review_records, file, indent=2)

print(f"\nTop-20 review receipt written to: {review_path}")

TOP-20 SKEPTICAL REVIEW
------------------------------------------------------------


,baseline_rank,content_id,review_action,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"194 days since update and 61,678 impressions in the last 90 days; score=100.00.",High rule confidence because both staleness and visibility are strong.,"the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
1,2,content_7368877ea310,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"194 days since update and 59,472 impressions in the last 90 days; score=100.00.",High rule confidence because both staleness and visibility are strong.,"the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
2,3,content_1bfaa38ff26c,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"194 days since update and 25,715 impressions in the last 90 days; score=100.00.",High rule confidence because both staleness and visibility are strong.,"the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
3,4,content_0a91db491d14,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"193 days since update and 13,299 impressions in the last 90 days; score=94.95.",High rule confidence because both staleness and visibility are strong.,"the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
4,5,content_5feee3994adb,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"194 days since update and 7,812 impressions in the last 90 days; score=90.04.",High rule confidence because both staleness and visibility are strong.,"the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
5,6,content_c2d929d83eaa,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"193 days since update and 7,558 impressions in the last 90 days; score=89.74.",High rule confidence because both staleness and visibility are strong.,"the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
6,7,content_b16bd7307b39,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"194 days since update and 4,590 impressions in the last 90 days; score=85.14.",High rule confidence because both staleness and visibility are strong.,"the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
7,8,content_fe16a55cd13d,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"194 days since update and 4,556 impressions in the last 90 days; score=85.07.",High rule confidence because both staleness and visibility are strong.,"the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
8,9,content_ecb6215e79fd,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"194 days since update and 4,429 impressions in the last 90 days; score=84.81.","Medium-high rule confidence; both conditions pass, but one component is less extreme.","the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."
9,10,content_928af3e22c80,REFRESH_CONTENT,STALE_VISIBLE_PAGE,"193 days since update and 1,697 impressions in the last 90 days; score=75.96.","Medium-high rule confidence; both conditions pass, but one component is less extreme.","the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important."



ONE-LINE REVIEW FOR EACH ROW
------------------------------------------------------------
Rank 1: Action=REFRESH_CONTENT | Why=194 days since update and 61,678 impressions in the last 90 days; score=100.00. | Wrong if=the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important.
Rank 2: Action=REFRESH_CONTENT | Why=194 days since update and 59,472 impressions in the last 90 days; score=100.00. | Wrong if=the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important.
Rank 3: Action=REFRESH_CONTENT | Why=194 days since update and 25,715 impressions in the last 90 days; score=100.00. | Wrong if=the topic may be seasonal, intentionally evergreen, already scheduled for refresh, or not commercially important.
Rank 4: Action=REFRESH_CONTENT | Why=193 days since update and 13,299 impressions in the last 90 days; score=94.95. | Wrong if=the topic may be seasonal, intentionally evergreen, alre

## 3. Top-20 review

I reviewed the first twenty recommendations because mistakes at the top of a ranked list are more important than mistakes near the bottom.

For every row, I report:

* The recommended action.
* Why the page appears in the queue.
* A confidence note.
* What information could make the recommendation wrong.

This is a skeptical review. The baseline only sees historical numeric signals. It cannot see the page title, topic accuracy, business priority, conversion value, recent editorial work that has not reached the data, seasonal demand, or whether a refresh is already planned.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-07 — SECTION 4
# Weak picks and leakage verification
# ============================================================

# ------------------------------------------------------------
# 1. Identify likely weak picks inside the top 20
# ------------------------------------------------------------

weak_pick_conditions = pd.Series(False, index=top_review.index)

if "avg_position" in top_review.columns:
    weak_pick_conditions |= (
        top_review["avg_position"].fillna(0) > 50
    )

if "sessions_90d" in top_review.columns:
    weak_pick_conditions |= (
        top_review["sessions_90d"].fillna(0) == 0
    )

if "ctr" in top_review.columns:
    weak_pick_conditions |= (
        top_review["ctr"].fillna(0) > 3
    )

weak_picks = top_review.loc[weak_pick_conditions].copy()

print("LIKELY WEAK PICKS IN THE TOP 20")
print("-" * 60)

if len(weak_picks) == 0:
    # The assignment expects skeptical inspection.
    # Show the lowest-scoring top recommendation when no obvious issue appears.
    weak_picks = top_review.tail(1).copy()

    print(
        "No row triggered the automatic weak-pick checks. "
        "The lowest-scoring row in the top 20 is shown for manual review."
    )

weak_columns = [
    column for column in [
        "baseline_rank",
        "content_id",
        "baseline_action_score",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "sessions_90d",
        "what_would_make_it_wrong",
    ]
    if column in weak_picks.columns
]

display(weak_picks[weak_columns])


# ------------------------------------------------------------
# 2. Explicit leakage check
# ------------------------------------------------------------

score_input_columns = [
    "days_since_last_update",
    "impressions_90d",
]

forbidden_columns = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "needs_refresh",
    "needs_ctr_fix",
    "is_quick_win",
    "refresh_flag",
    "action_flag",
    "content_id",
    "client_id",
}

leaked_inputs = sorted(
    set(score_input_columns).intersection(forbidden_columns)
)

assert leaked_inputs == [], (
    f"Leakage detected in score inputs: {leaked_inputs}"
)

assert set(score_input_columns) == {
    "days_since_last_update",
    "impressions_90d",
}, "Unexpected score inputs were used."

assert ranked_queue["baseline_action_score"].between(0, 100).all(), (
    "Scores must stay between 0 and 100."
)

assert ranked_queue["baseline_rank"].is_unique, (
    "Every ranked row must have a unique rank."
)

assert ranked_queue["content_id"].nunique() == len(ranked_queue), (
    "Expected one ranked row per content item."
)

assert queue_path.exists(), (
    "baseline_action_score.csv was not written."
)

print("\nLEAKAGE CHECK")
print("-" * 60)
print(f"Score inputs: {score_input_columns}")
print("Forbidden score inputs found: NONE")
print("Future-window inputs used: NO")
print("Label-derived score inputs used: NO")
print("Existing product flags used: NO")
print("IDs used as predictive inputs: NO")
print("Queue file exists: YES")
print("All verification assertions passed.")


# ------------------------------------------------------------
# 3. Final notebook summary
# ------------------------------------------------------------

print("\nFINAL ASSIGNMENT SUMMARY")
print("-" * 60)
print(f"Lane: Refresh / Content Opportunity Scoring")
print(f"Signal 1 verdict — staleness: {staleness_verdict}")
print(f"Signal 2 verdict — visibility: {volume_verdict}")
print(f"Rule reason code: {REASON_CODE}")
print(f"Primary action label: {REFRESH_ACTION}")
print(f"Rows ranked: {len(ranked_queue):,}")
print(f"Top rows reviewed: {len(top_review):,}")
print(f"Base rate: {base_rate:.2%}")
print(f"Precision@20: {metrics['precision_at_20']:.2%}")
print(f"Precision@50: {metrics['precision_at_50']:.2%}")
print(f"CSV written: {queue_path}")
print(f"Metrics JSON written: {metrics_path}")
print(f"Signal JSON written: {signal_json_path}")
print(f"Review JSON written: {review_path}")

LIKELY WEAK PICKS IN THE TOP 20
------------------------------------------------------------


,baseline_rank,content_id,baseline_action_score,days_since_last_update,impressions_90d,avg_position,ctr,sessions_90d,what_would_make_it_wrong
19,20,content_ba00ffc6318c,61.290,211,345,4.900,4.930,24,its click-through performance may already be healthy.



LEAKAGE CHECK
------------------------------------------------------------
Score inputs: ['days_since_last_update', 'impressions_90d']
Forbidden score inputs found: NONE
Future-window inputs used: NO
Label-derived score inputs used: NO
Existing product flags used: NO
IDs used as predictive inputs: NO
Queue file exists: YES
All verification assertions passed.

FINAL ASSIGNMENT SUMMARY
------------------------------------------------------------
Lane: Refresh / Content Opportunity Scoring
Signal 1 verdict — staleness: MIXED
Signal 2 verdict — visibility: CONFIRMED
Rule reason code: STALE_VISIBLE_PAGE
Primary action label: REFRESH_CONTENT
Rows ranked: 30,000
Top rows reviewed: 20
Base rate: 54.21%
Precision@20: 90.00%
Precision@50: 62.00%
CSV written: /content/flyrank-ml-internship-sunaina/work/outputs/baseline_action_score.csv
Metrics JSON written: /content/flyrank-ml-internship-sunaina/work/outputs/w04_baseline_metrics.json
Signal JSON written: /content/flyrank-ml-internship-sunaina/wo

## 4. Weak picks and leakage check

### Weak picks

The most likely weak recommendations are pages that pass the age and impression thresholds but have very deep average search positions, no meaningful sessions, or an already healthy click-through rate.

A page can also be a weak recommendation when:

* Its topic is seasonal.
* It is intentionally evergreen and still accurate.
* It has already been updated but the new update is not represented in the extract.
* Its impressions come from irrelevant searches.
* A technical SEO problem is more important than the page content.
* It has low business or conversion value.
* The page is already scheduled for deletion, consolidation, or redirection.

These limitations show why a ranked score should support human review rather than automatically trigger publication changes.

### Leakage check

The score uses only:

* `days_since_last_update`
* `impressions_90d`

The score does not use:

* `trend_direction`
* `trend_pct`
* `is_declining_label`
* Recent-versus-previous-window outcomes
* Existing FlyRank action flags
* Future information
* Raw identifiers as predictive features

`is_declining_label` is used only after scoring to calculate precision at K.

### Baseline conclusion

This baseline is intentionally simple. It creates a reproducible queue and a measurable Week 5 target. A later model will only be considered better if it improves ranking quality on an honest evaluation setup without using leaked or future information.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.